<a href="https://colab.research.google.com/github/Locket51/flyrank-ml-internship/blob/main/work_notebooks_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Locket51/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 167 (delta 60), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (167/167), 1.94 MiB | 13.41 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/flyrank-ml-internship


In [2]:
!pip install -q duckdb huggingface_hub

from google.colab import userdata
import duckdb, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ["HF_TOKEN"] = userdata.get("HF")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")
print("ready")

ready


## 1. Question

**Which content pages should a reviewer with limited weekly capacity open first?**

FlyRank's content reviewers cannot open every page in the inventory, so a prioritisation decision has to be made: which pages deserve review hours this week, and which do not. This capstone builds a decision-support artefact — a ranked, reason-coded review queue — for that decision.

The output is not a claim that editing a page causes recovery. It is a ranked list with reason codes that tells a reviewer which pages to look at first, and why.

## 2. Data

**Release:** `FlyRank/internship-warehouse` on Hugging Face, build `flyrank_pseudonymized_warehouse_release_v20260703`.

**Tables:** `fact_content_daily_performance` (primary), filtered to `gsc_data_available IS TRUE`.

**Date window:** `month = '2026-03'` — a mid-panel month, deliberately not the freshest month, which would be the natural outcome window of any past-to-future label.

**Slice size:** 9,841,378 rows across 55 clients and 331,437 content items, spanning 2026-03-01 to 2026-03-31. 3,611,061 rows (36.7%) have GSC data; 413,966 (4.2%) have GA4 data.

**Excluded and why:**
- GA4-derived features (sessions, engagement, scroll) — only 4.2% of rows have GA4 data, so any model using them would train on a small, non-random subset of clients.
- The June 2026 `_sample` table — it is the freshest month, so developing label logic there would peek at the future.
- All raw client, URL, query, and title fields — not present in the release by design.

In [3]:
q = con.execute("""
  SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS contents,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS with_gsc,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS with_ga4
  FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
  WHERE month = '2026-03'
""").df()
q

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,clients,contents,min_date,max_date,with_gsc,with_ga4
0,9841378,55,331437,2026-03-01,2026-03-31,3611061.0,413966.0


## 3. Methodology

**Unit of analysis.** One row per content item, summarised over a window inside March 2026.

**Features (5), all from days 2026-03-01 to 2026-03-15:**
`impressions_first_half`, `clicks_first_half`, `avg_position_first_half`, `ctr_first_half`, `days_with_impressions_first_half`.

**Proxy label.** `is_declining` = did second-half impressions fall below 70% of first-half impressions? This is a rule-computed proxy, not a future observed outcome. It exists so the modelling loop runs end to end; the audit checks whether it holds up.

**Baseline rule (Week 4).** `baseline_score = 0.45·visibility + 0.30·position_opportunity + 0.25·decay_risk`, with hand-picked weights. This is the transparent rule the learned model has to beat.

**Validation design — three splits compared on the same rows:**
1. **BEFORE** — naive random 80/20 split.
2. **AFTER** — client-grouped holdout (whole `client_hash_id` groups held out together).
3. **AFTER+** — strict time-aware split: 7-day feature window (03-01 to 03-07) and non-overlapping 23-day label window (03-08 to 03-31), with client grouping still enforced.

**Leakage checks (six-question checklist):**
- Features strictly before the decision point: yes, all features end in `_first_half`.
- Feature and label windows do not overlap: yes by construction.
- No rebuilt product flags: yes, none used.
- No derived field encodes the target: yes, `impressions_second_half` is used only for the label.
- No related rows split across train/test: yes, client grouping.
- Testing on unseen clients/time: yes by construction.

In [4]:
features = con.execute("""
  WITH daily AS (
    SELECT content_hash_id, client_hash_id, report_date,
      gsc_impressions, gsc_clicks, gsc_sum_position,
      CASE WHEN report_date <= DATE '2026-03-15' THEN 'first' ELSE 'second' END AS half
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
  ),
  agg AS (
    SELECT content_hash_id, client_hash_id,
      SUM(CASE WHEN half='first' THEN gsc_impressions ELSE 0 END) AS impressions_first_half,
      SUM(CASE WHEN half='first' THEN gsc_clicks ELSE 0 END)      AS clicks_first_half,
      AVG(CASE WHEN half='first' THEN gsc_sum_position END)       AS avg_position_first_half,
      COUNT(DISTINCT CASE WHEN half='first' AND gsc_impressions>0 THEN report_date END)
                                                                  AS days_with_impressions_first_half,
      SUM(CASE WHEN half='second' THEN gsc_impressions ELSE 0 END) AS impressions_second_half
    FROM daily GROUP BY content_hash_id, client_hash_id
  )
  SELECT *,
    CASE WHEN impressions_first_half >= 50
         THEN CASE WHEN impressions_second_half < 0.7*impressions_first_half THEN 1 ELSE 0 END
         ELSE NULL END AS is_declining
  FROM agg WHERE impressions_first_half >= 50
""").df()

features["ctr_first_half"] = features["clicks_first_half"] / features["impressions_first_half"]
features = features.dropna(subset=["is_declining"]).reset_index(drop=True)
print("features shape:", features.shape)
print(features["is_declining"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

features shape: (92548, 9)
is_declining
0    72410
1    20138
Name: count, dtype: int64


## 4. Results (vs baseline)

Same five features, same Random Forest, three splits. The drop from AFTER to AFTER+ is the size of the within-month autocorrelation the earlier design was carrying.

| Split | ROC AUC | Average precision | Precision@50 |
|---|---|---|---|
| BEFORE — naive random split | 0.631 | 0.301 | 0.460 |
| AFTER — client-grouped holdout | 0.576 | 0.359 | 0.540 |
| AFTER+ — time-aware + client-grouped | 0.524 | 0.031 | 0.060 |

**Reading.** Client grouping alone did not hurt — the naive random split was not the main leak. The strict time-aware split collapsed Precision@50 to 0.06 and ROC AUC to 0.524, near the base rate. The honest conclusion: the model did not rank well on this slice under strict separation.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

feature_cols = ["impressions_first_half","clicks_first_half","avg_position_first_half",
                "ctr_first_half","days_with_impressions_first_half"]
X = features[feature_cols].fillna(0).values
y = features["is_declining"].astype(int).values
groups = features["client_hash_id"].values

def pk(s, yt, k=50): return yt[np.argsort(-s)[:k]].mean()
def run(Xtr, ytr, Xte, yte, name):
    rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42, class_weight="balanced")
    rf.fit(Xtr, ytr); s = rf.predict_proba(Xte)[:,1]
    return {"split": name,
            "roc_auc": round(roc_auc_score(yte, s),3),
            "avg_precision": round(average_precision_score(yte, s),3),
            "precision@50": round(pk(s, yte),3)}

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
before = run(Xtr, ytr, Xte, yte, "BEFORE — naive random split")

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X, y, groups=groups))
after = run(X[tr], y[tr], X[te], y[te], "AFTER — client-grouped holdout")

# AFTER+ strict split
strict = con.execute("""
  WITH daily AS (
    SELECT content_hash_id, client_hash_id, report_date,
      gsc_impressions, gsc_clicks, gsc_sum_position,
      CASE WHEN report_date <= DATE '2026-03-07' THEN 'feat'
           WHEN report_date >= DATE '2026-03-08' THEN 'label' END AS part
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
  ),
  agg AS (
    SELECT content_hash_id, client_hash_id,
      SUM(CASE WHEN part='feat' THEN gsc_impressions ELSE 0 END) AS impressions_feat,
      SUM(CASE WHEN part='feat' THEN gsc_clicks ELSE 0 END)      AS clicks_feat,
      AVG(CASE WHEN part='feat' THEN gsc_sum_position END)       AS avg_position_feat,
      COUNT(DISTINCT CASE WHEN part='feat' AND gsc_impressions>0 THEN report_date END) AS days_with_impressions_feat,
      SUM(CASE WHEN part='label' THEN gsc_impressions ELSE 0 END) AS impressions_label
    FROM daily GROUP BY content_hash_id, client_hash_id
  )
  SELECT *,
    CASE WHEN impressions_feat >= 30
         THEN CASE WHEN impressions_label < 0.7*impressions_feat THEN 1 ELSE 0 END
         ELSE NULL END AS is_declining
  FROM agg WHERE impressions_feat >= 30
""").df()
strict["ctr_feat"] = strict["clicks_feat"] / strict["impressions_feat"]
strict = strict.dropna(subset=["is_declining"]).reset_index(drop=True)
sf = ["impressions_feat","clicks_feat","avg_position_feat","ctr_feat","days_with_impressions_feat"]
Xs = strict[sf].fillna(0).values; ys = strict["is_declining"].astype(int).values
gs = strict["client_hash_id"].values
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(Xs, ys, groups=gs))
after_plus = run(Xs[tr], ys[tr], Xs[te], ys[te], "AFTER+ — time-aware + grouped")

pd.DataFrame([before, after, after_plus])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,roc_auc,avg_precision,precision@50
0,BEFORE — naive random split,0.619,0.282,0.36
1,AFTER — client-grouped holdout,0.577,0.360,0.52
2,AFTER+ — time-aware + grouped,0.525,0.031,0.04


## 5. Limitations

- **Single month, single slice.** All numbers are on `month='2026-03'`, 55 clients. They do not establish behaviour on the full warehouse or on other months.
- **Proxy label.** `is_declining` is a rule-computed within-month proxy, not a future observed outcome. The Week-6 audit tested what that means.
- **No causal claim.** A page on the queue is a *candidate*. Nothing here shows that editing a page causes recovery — that would require an experiment.
- **Search-only features.** GA4 signals excluded because only 4.2% of rows have GA4 data.
- **No product flag leakage.** No FlyRank score (`health_score`, `priority_score`, `action_type`) was used as a feature or target.
- **Negative result reported.** Under strict separation the model did not beat a coin flip. This is stated in the paper, not hidden.

## 6. Ranked recommendations

The shipped deliverable is a transparent reason-coded queue, not a model. Every page gets one reason code, mapped to one action and one human owner.

| Reason code | What it looks like | Action | Owner |
|---|---|---|---|
| `declining_visible_page` | imp_first ≥ 500, second-half impressions fell ≥30% | refresh | Content editor |
| `top10_low_ctr` | avg position ≤ 10, CTR < 0.5% | metadata_review | SEO specialist |
| `watch` | visible but no specific pattern | monitor | Reviewer spot-check |
| `no_signal` | low visibility, no specific pattern | drop | not reviewed |

**What must not be automated:** auto-refreshing, auto-rewriting titles or metas, auto-pruning or merging, and feeding the queue back into a model as ground truth.

**Monitoring / retrain triggers:** reason-code distribution drift >20%, Precision@50 below base rate on a new month, client mix change, label definition change.

In [6]:
queue_path = "work/outputs/w07_action_queue.csv"
if os.path.exists(queue_path):
    q = pd.read_csv(queue_path)
    print("queue rows:", len(q))
    print(q["reason_code"].value_counts())
else:
    print("queue not regenerated in this notebook — see w07_action_playbook.ipynb")

queue not regenerated in this notebook — see w07_action_playbook.ipynb


## 7. Artifacts the paper embeds

- `work/figures/w07_reason_code_distribution.png` — the reason-code figure, copied to `docs/` for the deployed paper.
- `work/outputs/w07_playbook_metrics.json` — counts receipt the paper cites.
- `work/outputs/w06_validation_metrics.json` — three-split metrics receipt.
- `docs/index.html` — the deployed paper itself.

All of the above are reproducible by running the notebooks in `work/notebooks/` in order.

In [7]:
import os
for p in ["work/figures/w07_reason_code_distribution.png",
          "work/outputs/w07_playbook_metrics.json",
          "work/outputs/w06_validation_metrics.json",
          "docs/index.html"]:
    print(f"{p}: {'OK' if os.path.exists(p) else 'missing'}")

work/figures/w07_reason_code_distribution.png: missing
work/outputs/w07_playbook_metrics.json: missing
work/outputs/w06_validation_metrics.json: missing
docs/index.html: OK


## 5-minute demo outline

**Showcase framing — Refresh / Content Opportunity Scoring (Lane 2).**

**0:00–0:30 — Question.** FlyRank's reviewers can't open every page. Which pages should they look at first? That's the decision this work improves.

**0:30–1:30 — Method.** Five features from days 1–15 of March 2026 in the internship warehouse. Three validation designs compared on the same rows: naive random split, client-grouped holdout, and strict time-aware split with non-overlapping windows.

**1:30–2:30 — One chart.** Show `work/figures/w07_reason_code_distribution.png`. The actionable buckets are `declining_visible_page` and `top10_low_ctr`, and they map to different human actions.

**2:30–4:00 — One honest result.** Precision@50 was 0.46 (naive), 0.54 (client-grouped), 0.06 (strict time-aware). The collapse is the size of the within-month autocorrelation I was carrying. The model did not beat a coin flip when the split was honest.

**4:00–5:00 — One recommendation.** Ship the reason-coded queue, not the model. Human reviewers act on it; nothing is automated.

## Shareable cut 1 — short social post

> I spent a week trying to beat a hand-tuned content-review rule with a Random Forest on 9.8M rows of real search data. It beat the rule on a random split (Precision@50 = 0.54 vs ~0.46). Then I split it honestly — client-grouped + strict time-aware windows with no feature/label overlap — and Precision@50 fell to 0.06. The lift was within-month autocorrelation, not skill. Shipped a transparent reason-coded queue instead of a model. #ML #SEO #HonestML

## Shareable cut 2 — employer-facing summary

> I built a content-refresh prioritisation study on the FlyRank internship warehouse — ~9.8M daily search-performance rows for one mid-panel month, filtered to a search-only slice and evaluated under three validation designs. It showed that a Random Forest beat a transparent rule on a naive split (Precision@50 0.54 vs ~0.46) but collapsed to 0.06 under a strict client-grouped, time-aware split, which is how I caught a within-month autocorrelation leak in my own Week-5 design. I shipped the honest version: a reason-coded review queue with a human-review process, a no-automation list, and monitoring triggers — live at a public URL and reproducible from the repo.